In [4]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from upath import UPath

import wave_buoys_viewer as wbv

data_dir = UPath("../data")

data_fake_path = data_dir / "wave_buoys_data_2_years.parquet"
data_path = data_dir / "wave_buoys_data.parquet"

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
# If the data file already exists, read the latest data from it
# NOTE: in the future we might want to only load the latest rows by date
if data_fake_path.exists():
    data = pd.read_parquet(str(data_fake_path))
else:
    data = pd.DataFrame()

# Scrap wave buoy data
campaign_id = "camp=06403"
new_data = wbv.scrap.scrap_wave_buoy_data(campaign_id=campaign_id)

# Append new data and keep only the latest for duplicate campaign_id/datetime
data = pd.concat([data, new_data], ignore_index=True)
data = data.drop_duplicates(subset=["campaign_id", "datetime"], keep='last')

# Create directory if it does not exist
data_path.parent.mkdir(parents=True, exist_ok=True)

# Save data to parquet file
data.to_parquet(str(data_path), index=False)